# 도보 네트워크 속성 vs 외부 RAW 데이터 overlap 검증

이 노트북의 목적은 외부 RAW 데이터를 바로 새로운 layer/score로 연결하는 것이 아니라, 도보 네트워크 자체에 이미 존재하는 `횡단보도`, `육교`, `터널`, `공원,녹지` 속성이 외부 RAW 데이터와 공간적으로 같은 현실을 가리키는지 확인하는 것입니다.

외부 RAW 데이터는 다음 세 가지 관점으로 분류합니다.

1. 기존 도보 네트워크 속성의 신뢰도 검증용
2. 기존 속성에 없는 구간을 보강하는 layer 후보
3. 기존 데이터와 충돌하거나 품질이 낮아 보류할 데이터

판정은 단순한 겹침/안 겹침이 아니라 `기존 네트워크 속성만으로 충분함`, `외부 RAW로 검증 가능`, `외부 RAW로 보강 가능`, `기존 속성과 외부 RAW가 충돌함`, `좌표계/geometry 문제로 판단 보류` 중 하나로 남깁니다.

## 검증 대상

| 도보 네트워크 속성 | 외부 RAW 후보 | 1차 거리 기준 |
|---|---|---:|
| `횡단보도` | `서울시 대로변 횡단보도 위치정보.csv` | 20m |
| `육교` | `서울시 육교 공간정보.csv` | 20m |
| `터널` | `국토교통부_전국도로터널정보표준데이터_20251231.csv` | 50m |
| `공원,녹지` | `서울시 주요 공원현황.csv`, `전국도시공원정보표준데이터.csv`, `서울시 공원 및 사유지수목`, `서울시 둘레길/문화길` | 50m |

`공원,녹지` 후보 중 일부는 polygon이 아니라 대표점/수목점/길 선형이므로, 결과가 낮게 나와도 곧바로 충돌로 보지 않고 geometry 한계를 같이 봅니다.

In [ ]:
from pathlib import Path
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely import wkt
from shapely.geometry import LineString, Point

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "raw":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "analysis":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "src" / "data" / "raw"
TABLE_DIR = PROJECT_ROOT / "analysis" / "tables" / "raw"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

WGS84 = "EPSG:4326"
METRIC_CRS = "EPSG:5179"

RAW_DIR, TABLE_DIR

In [ ]:
def read_csv_auto(path: Path, **kwargs) -> pd.DataFrame:
    """Read Korean public-data CSV files with a small encoding fallback list."""
    errors = []
    for encoding in ("utf-8-sig", "cp949", "euc-kr"):
        try:
            return pd.read_csv(path, encoding=encoding, **kwargs)
        except UnicodeDecodeError as exc:
            errors.append(f"{encoding}: {exc}")
    raise UnicodeDecodeError("csv", b"", 0, 1, f"failed encodings: {errors}")


def to_flag(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).astype(float).ne(0)
    normalized = series.fillna("").astype(str).str.strip().str.lower()
    truthy = {"1", "y", "yes", "true", "t", "유", "있음", "해당", "o"}
    falsy = {"", "0", "n", "no", "false", "f", "무", "없음", "x", "nan", "none"}
    return normalized.map(lambda value: False if value in falsy else value in truthy or value != "")


def parse_wkt_or_none(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None
    try:
        return wkt.loads(text)
    except Exception:
        return None


def gdf_from_wkt(df: pd.DataFrame, wkt_col: str, crs=WGS84) -> gpd.GeoDataFrame:
    out = df.copy()
    out["geometry"] = out[wkt_col].map(parse_wkt_or_none)
    out = out[out["geometry"].notna()].copy()
    return gpd.GeoDataFrame(out, geometry="geometry", crs=crs)


def gdf_from_lon_lat(df: pd.DataFrame, lon_col: str, lat_col: str, crs=WGS84) -> gpd.GeoDataFrame:
    out = df.copy()
    out[lon_col] = pd.to_numeric(out[lon_col], errors="coerce")
    out[lat_col] = pd.to_numeric(out[lat_col], errors="coerce")
    out = out[out[lon_col].notna() & out[lat_col].notna()].copy()
    out["geometry"] = [Point(xy) for xy in zip(out[lon_col], out[lat_col])]
    return gpd.GeoDataFrame(out, geometry="geometry", crs=crs)


def seoul_bbox_filter(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    wgs = gdf.to_crs(WGS84)
    mask = (
        wgs.geometry.x.between(126.7, 127.3)
        & wgs.geometry.y.between(37.35, 37.75)
    )
    return gdf.loc[mask.values].copy()

## 1. 도보 네트워크 edge와 내부 속성 확인

도보 네트워크 원본에는 `LINK` 행과 `NODE` 행이 함께 들어 있습니다. 엑셀에서 컬럼 전체를 합산하면 `NODE`에 붙은 횡단보도/육교 값까지 포함되고, 공간 overlap 검증에서는 `링크 WKT`가 있는 `LINK` edge만 사용할 수 있습니다.

따라서 먼저 `전체 원본 합계`, `LINK 합계`, `NODE 합계`를 나눠 보고, 이후 모든 거리 계산은 `LINK` edge를 meter 단위 좌표계인 EPSG:5179로 변환해서 수행합니다.

In [ ]:
WALK_FILE = RAW_DIR / "서울시 자치구별 도보 네트워크 공간정보.csv"
WALK_USECOLS = [
    "노드링크 유형", "노드 WKT", "노드 ID", "링크 WKT", "링크 ID", "링크 길이", "시군구명", "읍면동명",
    "고가도로", "지하철네트워크", "교량", "터널", "육교", "횡단보도", "공원,녹지", "건물내",
]

walk_raw = read_csv_auto(WALK_FILE, usecols=lambda col: col in WALK_USECOLS)

ATTR_COLUMNS = ["고가도로", "지하철네트워크", "교량", "터널", "육교", "횡단보도", "공원,녹지", "건물내"]
raw_attr_rows = []
for col in ATTR_COLUMNS:
    values = pd.to_numeric(walk_raw[col], errors="coerce").fillna(0)
    link_values = pd.to_numeric(walk_raw.loc[walk_raw["노드링크 유형"].eq("LINK"), col], errors="coerce").fillna(0)
    node_values = pd.to_numeric(walk_raw.loc[walk_raw["노드링크 유형"].eq("NODE"), col], errors="coerce").fillna(0)
    raw_attr_rows.append({
        "속성": col,
        "전체 원본 합계(Excel SUM과 동일)": int(values.sum()),
        "LINK edge 합계(공간검증 대상)": int(link_values.sum()),
        "NODE 합계(edge geometry 없음)": int(node_values.sum()),
    })

raw_attr_split = pd.DataFrame(raw_attr_rows)
display(raw_attr_split)

walk_edges = gdf_from_wkt(walk_raw, "링크 WKT")
walk_edges = walk_edges[walk_edges.geometry.geom_type.isin(["LineString", "MultiLineString"])].copy()
walk_edges = walk_edges.rename(columns={"링크 ID": "walk_edge_id"})
walk_edges["walk_edge_id"] = walk_edges["walk_edge_id"].astype(str)

SPECIAL_ATTRS = {
    "crosswalk": {"column": "횡단보도", "label": "횡단보도"},
    "overpass": {"column": "육교", "label": "육교"},
    "subway": {"column": "지하철네트워크", "label": "지하철네트워크"},
    "tunnel": {"column": "터널", "label": "터널"},
    "green": {"column": "공원,녹지", "label": "공원/녹지"},
    "bridge": {"column": "교량", "label": "교량"},
    "elevated": {"column": "고가도로", "label": "고가도로"},
    "indoor": {"column": "건물내", "label": "건물내"},
}

for attr in SPECIAL_ATTRS.values():
    col = attr["column"]
    if col in walk_edges.columns:
        walk_edges[f"is_{col}"] = to_flag(walk_edges[col])

walk_edges_m = walk_edges.to_crs(METRIC_CRS)

preview_cols = ["walk_edge_id", "시군구명", "링크 길이"] + [
    v["column"] for v in SPECIAL_ATTRS.values() if v["column"] in walk_edges.columns
]
flag_cols = [f"is_{v['column']}" for v in SPECIAL_ATTRS.values() if f"is_{v['column']}" in walk_edges.columns]

print(f"전체 도보 edge 수: {len(walk_edges):,}")
display(pd.DataFrame([
    {
        "속성": spec["label"],
        "컬럼": spec["column"],
        "flag_edge_count": int(walk_edges[f"is_{spec['column']}"].sum()),
    }
    for spec in SPECIAL_ATTRS.values()
    if f"is_{spec['column']}" in walk_edges.columns
]).sort_values("flag_edge_count", ascending=False))

display(walk_edges.loc[walk_edges[flag_cols].any(axis=1), preview_cols].head(10))

In [ ]:
walk_attr_summary = []
for key, spec in SPECIAL_ATTRS.items():
    col = spec["column"]
    flag_col = f"is_{col}"
    if flag_col not in walk_edges.columns:
        continue
    edge_count = int(walk_edges[flag_col].sum())
    length_m = pd.to_numeric(walk_edges.loc[walk_edges[flag_col], "링크 길이"], errors="coerce").sum()
    walk_attr_summary.append({
        "attr_key": key,
        "network_attr": spec["label"],
        "network_flag_edge_count": edge_count,
        "network_flag_length_m": round(float(length_m), 1),
        "network_flag_edge_ratio": edge_count / len(walk_edges) if len(walk_edges) else np.nan,
    })

walk_attr_summary = pd.DataFrame(walk_attr_summary).sort_values("network_flag_edge_count", ascending=False)
walk_attr_summary.to_csv(TABLE_DIR / "walk_edge_special_attr_summary_recomputed.csv", index=False, encoding="utf-8-sig")
walk_attr_summary

## 2. 외부 RAW geometry 만들기

외부 RAW는 세 가지 geometry 수준으로 나뉩니다.

- `line_wkt`: 도보 네트워크와 가장 직접 비교 가능
- `line_from_endpoints`: 시작/종료 좌표로 만든 선형이라 비교 가능하지만 도로 중심선일 수 있음
- `point_proxy`: 대표점/시설점이므로 공원·녹지 edge 검증에는 보조 근거로만 사용

In [ ]:
def load_line_wkt_raw(file_name: str, dataset_key: str, dataset_label: str) -> gpd.GeoDataFrame:
    df = read_csv_auto(RAW_DIR / file_name)
    gdf = gdf_from_wkt(df, "링크 WKT")
    gdf = gdf[gdf.geometry.geom_type.isin(["LineString", "MultiLineString"])].copy()
    gdf["raw_dataset"] = dataset_key
    gdf["raw_dataset_label"] = dataset_label
    gdf["geometry_level"] = "line_wkt"
    gdf["raw_feature_id"] = dataset_key + ":" + gdf.index.astype(str)
    return gdf


def load_tunnel_raw(sidewalk_only: bool = False) -> gpd.GeoDataFrame:
    df = read_csv_auto(RAW_DIR / "국토교통부_전국도로터널정보표준데이터_20251231.csv")
    if "시도명" in df.columns:
        df = df[df["시도명"].astype(str).str.contains("서울", na=False)].copy()
    if sidewalk_only:
        df["터널보도폭"] = pd.to_numeric(df.get("터널보도폭"), errors="coerce").fillna(0)
        df = df[df["터널보도폭"].gt(0)].copy()
    for col in ["터널시작점경도", "터널시작점위도", "터널종료점경도", "터널종료점위도"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["터널시작점경도", "터널시작점위도", "터널종료점경도", "터널종료점위도"]).copy()
    df["geometry"] = [
        LineString([(row["터널시작점경도"], row["터널시작점위도"]), (row["터널종료점경도"], row["터널종료점위도"])])
        for _, row in df.iterrows()
    ]
    gdf = gpd.GeoDataFrame(df, geometry="geometry", crs=WGS84)
    gdf["raw_dataset"] = "road_tunnel_seoul_sidewalk" if sidewalk_only else "road_tunnel_seoul_all"
    gdf["raw_dataset_label"] = "국토교통부 전국도로터널정보(서울, 보도폭>0)" if sidewalk_only else "국토교통부 전국도로터널정보(서울 전체)"
    gdf["geometry_level"] = "line_from_endpoints"
    gdf["raw_feature_id"] = gdf["raw_dataset"] + ":" + gdf.index.astype(str)
    return gdf


def load_point_raw(file_name: str, dataset_key: str, dataset_label: str, lon_col: str, lat_col: str) -> gpd.GeoDataFrame:
    df = read_csv_auto(RAW_DIR / file_name)
    gdf = gdf_from_lon_lat(df, lon_col, lat_col)
    gdf = seoul_bbox_filter(gdf)
    gdf["raw_dataset"] = dataset_key
    gdf["raw_dataset_label"] = dataset_label
    gdf["geometry_level"] = "point_proxy"
    gdf["raw_feature_id"] = dataset_key + ":" + gdf.index.astype(str)
    return gdf


external_layers = {
    "seoul_crosswalk": load_line_wkt_raw("서울시 대로변 횡단보도 위치정보.csv", "seoul_crosswalk", "서울시 대로변 횡단보도"),
    "seoul_overpass": load_line_wkt_raw("서울시 육교 공간정보.csv", "seoul_overpass", "서울시 육교 공간정보"),
    "road_tunnel_seoul_all": load_tunnel_raw(sidewalk_only=False),
    "road_tunnel_seoul_sidewalk": load_tunnel_raw(sidewalk_only=True),
    "seoul_major_park": load_point_raw("서울시 주요 공원현황.csv", "seoul_major_park", "서울시 주요 공원현황", "X좌표(WGS84)", "Y좌표(WGS84)"),
    "national_city_park": load_point_raw("전국도시공원정보표준데이터.csv", "national_city_park", "전국도시공원정보", "경도", "위도"),
    "seoul_private_tree": load_point_raw("서울시 공원 및 사유지수목 위치정보 (좌표계_ WGS1984).csv", "seoul_private_tree", "서울시 공원 및 사유지수목", "경도", "위도"),
    "seoul_trail_line": load_point_raw("서울시 둘레길 선형 위치정보 (좌표계_ WGS1984).csv", "seoul_trail_line", "서울시 둘레길 선형", "경도", "위도"),
    "seoul_culture_trail_line": load_point_raw("서울시 문화길 선형 위치정보 (좌표계_ WGS1984).csv", "seoul_culture_trail_line", "서울시 문화길 선형", "경도", "위도"),
}

external_layer_summary = pd.DataFrame([
    {
        "raw_dataset": key,
        "raw_dataset_label": gdf["raw_dataset_label"].iloc[0] if len(gdf) else key,
        "geometry_level": gdf["geometry_level"].iloc[0] if len(gdf) else "empty",
        "feature_count": len(gdf),
        "geometry_types": ", ".join(sorted(gdf.geometry.geom_type.unique())) if len(gdf) else "empty",
    }
    for key, gdf in external_layers.items()
])
external_layer_summary.to_csv(TABLE_DIR / "external_raw_geometry_summary.csv", index=False, encoding="utf-8-sig")
external_layer_summary

### 이 표를 왜 보나

`external_layer_summary`는 overlap 결과가 아니라 **검증 재료 목록**입니다. 여기서 먼저 볼 것은 `geometry_level`입니다.

- `line_wkt`: 외부 RAW가 선형입니다. 도보 `LINK` edge와 선-선 거리 또는 교차 여부로 비교할 수 있습니다.
- `line_from_endpoints`: 시작/종료점으로 만든 선형입니다. 비교는 가능하지만 실제 선형이 단순화되어 있을 수 있습니다.
- `point_proxy`: 대표점입니다. 도보 edge와 바로 겹침을 판정하기 약하므로 보조 근거 또는 후보 탐색용으로만 사용합니다.

따라서 이 표의 결론은 `몇 개 있다`가 아니라, **이 RAW로 어떤 방식의 공간 검증을 할 수 있는가**입니다.

In [ ]:
overlap_method_plan = pd.DataFrame([
    {
        "network_attr": "횡단보도",
        "network_geometry": "LINK edge + NODE point 분리",
        "external_raw": "서울시 대로변 횡단보도 위치정보",
        "overlap_method": "LINK는 선-선 20m 근접, NODE는 점-선/점-점 20m 근접을 별도 검증",
        "meaning": "edge 속성 신뢰도와 횡단보도 지점 신뢰도를 분리해서 본다",
        "decision_use": "검증용 + 누락 보강 후보",
    },
    {
        "network_attr": "육교",
        "network_geometry": "LINK edge + NODE point 분리",
        "external_raw": "서울시 육교 공간정보",
        "overlap_method": "LINK는 선-선 20m 근접, NODE는 육교 endpoint/대표점과 20m 근접 검증",
        "meaning": "육교가 길 구간 속성인지 시설 지점 속성인지 분리한다",
        "decision_use": "검증용 + accessibility/risk layer 후보",
    },
    {
        "network_attr": "터널",
        "network_geometry": "LINK edge",
        "external_raw": "국토교통부 전국도로터널정보(서울, 보도폭>0)",
        "overlap_method": "서울 필터 + 터널보도폭>0 필터 후, 터널 시작/종료점 선형과 도보 터널 edge를 50m 근접으로 비교",
        "meaning": "보도폭이 있는 도로터널 RAW가 보행 네트워크의 터널 의미와 같은지 확인한다",
        "decision_use": "검증용, 불일치가 크면 의미 충돌/보류",
    },
    {
        "network_attr": "공원/녹지",
        "network_geometry": "LINK edge",
        "external_raw": "주요 공원, 전국도시공원, 수목, 둘레길/문화길",
        "overlap_method": "현재 RAW는 대부분 point_proxy라 edge와 50~100m 근접만 확인. polygon 확보 전까지 보조 근거",
        "meaning": "공원 내부 edge인지, 공원 근처 edge인지, 산책길 후보인지 구분해야 한다",
        "decision_use": "보류 또는 보강 후보. polygon/OSM 녹지와 재검증 필요",
    },
    {
        "network_attr": "교량",
        "network_geometry": "LINK edge",
        "external_raw": "현재 직접 대응 RAW 없음",
        "overlap_method": "직접 overlap 불가. 하천 GeoJSON과 교량 edge 위치를 비교하는 간접 검증만 가능",
        "meaning": "교량 속성 자체 검증보다는 하천 crossing 여부 확인에 가깝다",
        "decision_use": "직접 검증 보류",
    },
    {
        "network_attr": "고가도로/건물내/지하철네트워크",
        "network_geometry": "LINK edge",
        "external_raw": "현재 직접 대응 RAW 부족",
        "overlap_method": "대응 RAW가 없으면 overlap 검증하지 않고 내부 속성으로만 관리",
        "meaning": "외부 RAW로 새 score를 만들 단계가 아니라 데이터 의미만 기록한다",
        "decision_use": "기존 네트워크 속성 우선 사용, 외부 검증 보류",
    },
])

overlap_method_plan.to_csv(TABLE_DIR / "walk_edge_external_overlap_method_plan.csv", index=False, encoding="utf-8-sig")
overlap_method_plan

## 3. overlap 판정 함수

각 비교는 두 방향으로 봅니다.

1. 도보 네트워크에서 해당 속성이 켜진 edge 중 외부 RAW와 가까운 비율
2. 외부 RAW feature 중 도보 네트워크의 같은 속성 edge와 가까운 비율

첫 번째가 낮으면 기존 속성의 의미나 좌표계 문제가 의심되고, 두 번째가 낮으면 외부 RAW가 다른 범위를 말하거나 보강 후보일 수 있습니다.

In [ ]:
VALIDATION_PAIRS = [
    {
        "attr_key": "crosswalk",
        "network_column": "횡단보도",
        "raw_keys": ["seoul_crosswalk"],
        "distance_m": 20,
    },
    {
        "attr_key": "overpass",
        "network_column": "육교",
        "raw_keys": ["seoul_overpass"],
        "distance_m": 20,
    },
    {
        "attr_key": "tunnel",
        "network_column": "터널",
        "raw_keys": ["road_tunnel_seoul_sidewalk"],
        "distance_m": 50,
    },
    {
        "attr_key": "green",
        "network_column": "공원,녹지",
        "raw_keys": ["seoul_major_park", "national_city_park", "seoul_private_tree", "seoul_trail_line", "seoul_culture_trail_line"],
        "distance_m": 50,
    },
]


def combine_raw(raw_keys: list[str]) -> gpd.GeoDataFrame:
    frames = [external_layers[key] for key in raw_keys if key in external_layers and len(external_layers[key])]
    if not frames:
        return gpd.GeoDataFrame(columns=["raw_feature_id", "raw_dataset", "raw_dataset_label", "geometry_level", "geometry"], geometry="geometry", crs=WGS84)
    return pd.concat(frames, ignore_index=True).pipe(gpd.GeoDataFrame, geometry="geometry", crs=WGS84)


def sjoin_nearest_unique(left: gpd.GeoDataFrame, right: gpd.GeoDataFrame, max_distance: float, left_id: str, right_cols: list[str]) -> pd.DataFrame:
    if len(left) == 0 or len(right) == 0:
        return pd.DataFrame(columns=[left_id, "distance_m"] + right_cols)
    joined = gpd.sjoin_nearest(
        left[[left_id, "geometry"]].to_crs(METRIC_CRS),
        right[right_cols + ["geometry"]].to_crs(METRIC_CRS),
        how="left",
        max_distance=max_distance,
        distance_col="distance_m",
    )
    joined = joined.sort_values([left_id, "distance_m"], na_position="last")
    return joined.drop_duplicates(left_id)


def classify_decision(row: pd.Series) -> str:
    if row["raw_feature_count"] == 0:
        return "좌표계/geometry 문제로 판단 보류"
    if row["has_point_proxy_only"]:
        if row["network_flag_match_rate"] >= 0.5:
            return "외부 RAW로 검증 가능"
        return "좌표계/geometry 문제로 판단 보류"
    if row["network_flag_match_rate"] >= 0.8 and row["raw_to_network_attr_match_rate"] >= 0.8:
        return "기존 네트워크 속성만으로 충분함"
    if row["network_flag_match_rate"] >= 0.5 and row["raw_only_candidate_count"] > 0:
        return "외부 RAW로 보강 가능"
    if row["network_flag_match_rate"] < 0.2 and row["raw_to_network_attr_match_rate"] < 0.2:
        return "기존 속성과 외부 RAW가 충돌함"
    return "외부 RAW로 검증 가능"


def validate_pair(pair: dict) -> tuple[dict, pd.DataFrame, pd.DataFrame]:
    attr_key = pair["attr_key"]
    network_col = pair["network_column"]
    flag_col = f"is_{network_col}"
    raw = combine_raw(pair["raw_keys"])
    distance_m = pair["distance_m"]

    flagged = walk_edges[walk_edges[flag_col]].copy()
    raw = raw.copy()

    flagged_to_raw = sjoin_nearest_unique(
        flagged,
        raw,
        distance_m,
        "walk_edge_id",
        ["raw_feature_id", "raw_dataset", "raw_dataset_label", "geometry_level"],
    )
    network_matched = flagged_to_raw["raw_feature_id"].notna()

    edge_lookup = walk_edges[["walk_edge_id", flag_col, "시군구명", "읍면동명", "geometry"]].copy()
    raw_to_walk = sjoin_nearest_unique(
        raw,
        edge_lookup,
        distance_m,
        "raw_feature_id",
        ["walk_edge_id", flag_col, "시군구명", "읍면동명"],
    )
    raw_near_walk = raw_to_walk["walk_edge_id"].notna()
    raw_near_same_attr = raw_near_walk & raw_to_walk[flag_col].fillna(False)

    raw_geometry_levels = sorted(raw["geometry_level"].dropna().unique().tolist()) if len(raw) else []
    summary = {
        "attr_key": attr_key,
        "network_column": network_col,
        "raw_keys": ", ".join(pair["raw_keys"]),
        "distance_m": distance_m,
        "network_flag_edge_count": len(flagged),
        "raw_feature_count": len(raw),
        "network_flag_matched_count": int(network_matched.sum()),
        "network_flag_match_rate": float(network_matched.mean()) if len(flagged) else np.nan,
        "raw_near_any_walk_count": int(raw_near_walk.sum()),
        "raw_to_network_attr_matched_count": int(raw_near_same_attr.sum()),
        "raw_to_network_attr_match_rate": float(raw_near_same_attr.mean()) if len(raw) else np.nan,
        "network_only_count": int((~network_matched).sum()),
        "raw_only_candidate_count": int((raw_near_walk & ~raw_to_walk[flag_col].fillna(False)).sum()),
        "raw_without_walk_nearby_count": int((~raw_near_walk).sum()),
        "raw_geometry_levels": ", ".join(raw_geometry_levels),
        "has_point_proxy_only": raw_geometry_levels == ["point_proxy"],
    }
    summary["decision"] = classify_decision(pd.Series(summary))

    unmatched_network = flagged_to_raw[flagged_to_raw["raw_feature_id"].isna()].copy()
    raw_only = raw_to_walk[raw_near_walk & ~raw_to_walk[flag_col].fillna(False)].copy()
    unmatched_network["attr_key"] = attr_key
    raw_only["attr_key"] = attr_key
    return summary, unmatched_network, raw_only

In [ ]:
summaries = []
unmatched_network_samples = []
raw_only_samples = []

for pair in VALIDATION_PAIRS:
    summary, unmatched_network, raw_only = validate_pair(pair)
    summaries.append(summary)
    unmatched_network_samples.append(unmatched_network.head(200))
    raw_only_samples.append(raw_only.head(200))

overlap_summary = pd.DataFrame(summaries)
unmatched_network_sample = pd.concat(unmatched_network_samples, ignore_index=True) if unmatched_network_samples else pd.DataFrame()
raw_only_candidate_sample = pd.concat(raw_only_samples, ignore_index=True) if raw_only_samples else pd.DataFrame()

overlap_summary.to_csv(TABLE_DIR / "walk_edge_external_raw_overlap_summary.csv", index=False, encoding="utf-8-sig")
unmatched_network_sample.to_csv(TABLE_DIR / "walk_edge_external_raw_network_only_sample.csv", index=False, encoding="utf-8-sig")
raw_only_candidate_sample.to_csv(TABLE_DIR / "walk_edge_external_raw_raw_only_candidate_sample.csv", index=False, encoding="utf-8-sig")

overlap_summary

## 4. 자치구별로 어디가 어긋나는지 보기

요약 비율만으로 결론 내리지 말고, `network_only`와 `raw_only_candidate`가 특정 자치구에 몰리는지 확인합니다. 특정 구에만 몰리면 좌표계/수집범위/갱신일 문제일 가능성이 큽니다.

In [ ]:
def gu_mismatch_table(pair: dict) -> pd.DataFrame:
    attr_key = pair["attr_key"]
    network_col = pair["network_column"]
    flag_col = f"is_{network_col}"
    raw = combine_raw(pair["raw_keys"])
    flagged = walk_edges[walk_edges[flag_col]].copy()

    flagged_to_raw = sjoin_nearest_unique(
        flagged,
        raw,
        pair["distance_m"],
        "walk_edge_id",
        ["raw_feature_id", "raw_dataset", "raw_dataset_label", "geometry_level"],
    )
    base = flagged[["walk_edge_id", "시군구명"]].merge(flagged_to_raw[["walk_edge_id", "raw_feature_id"]], on="walk_edge_id", how="left")
    out = base.assign(network_only=base["raw_feature_id"].isna()).groupby("시군구명", dropna=False).agg(
        network_flag_edge_count=("walk_edge_id", "count"),
        network_only_count=("network_only", "sum"),
    ).reset_index()
    out["network_only_rate"] = out["network_only_count"] / out["network_flag_edge_count"]
    out["attr_key"] = attr_key
    return out.sort_values("network_only_rate", ascending=False)


gu_mismatch = pd.concat([gu_mismatch_table(pair) for pair in VALIDATION_PAIRS], ignore_index=True)
gu_mismatch.to_csv(TABLE_DIR / "walk_edge_external_raw_overlap_by_gu.csv", index=False, encoding="utf-8-sig")
gu_mismatch.head(30)

## 5. 빠른 지도 확인

샘플 지도는 수치 결과가 이상할 때만 봅니다. 특히 `공원,녹지`처럼 대표점 기반 RAW는 지도에서 실제 공원/길과 edge가 얼마나 가까운지 육안 확인이 필요합니다.

In [ ]:
def quick_overlap_map(attr_key: str, raw_keys: list[str] | None = None, sample_n: int = 200):
    import folium

    pair = next(item for item in VALIDATION_PAIRS if item["attr_key"] == attr_key)
    network_col = pair["network_column"]
    flag_col = f"is_{network_col}"
    raw = combine_raw(raw_keys or pair["raw_keys"])
    flagged = walk_edges[walk_edges[flag_col]].sample(min(sample_n, int(walk_edges[flag_col].sum())), random_state=42)
    raw_sample = raw.sample(min(sample_n, len(raw)), random_state=42) if len(raw) else raw

    m = folium.Map(location=[37.56, 126.98], zoom_start=12, tiles="CartoDB positron")

    folium.GeoJson(
        flagged.to_crs(WGS84).__geo_interface__,
        name=f"network {network_col} sample",
        style_function=lambda _: {"color": "#2563eb", "weight": 3, "opacity": 0.75},
    ).add_to(m)

    if len(raw_sample):
        folium.GeoJson(
            raw_sample.to_crs(WGS84).__geo_interface__,
            name="external raw sample",
            style_function=lambda _: {"color": "#dc2626", "weight": 2, "opacity": 0.65},
            marker=folium.CircleMarker(radius=3, color="#dc2626", fill=True, fill_opacity=0.8),
        ).add_to(m)

    folium.LayerControl().add_to(m)
    return m


# 예시: quick_overlap_map("crosswalk")
# 예시: quick_overlap_map("green", raw_keys=["seoul_major_park", "national_city_park"])

## 6. 횡단보도/육교 NODE 검증

`횡단보도`와 `육교`는 `LINK`뿐 아니라 `NODE`에도 값이 있습니다. 이 값은 길 구간이 아니라 지점 속성에 가깝기 때문에, edge overlap과 분리해서 점-선 근접으로 검증합니다.

- 네트워크 NODE -> 외부 RAW 선형: 기존 NODE 속성이 외부 RAW와 가까운가
- 외부 RAW 선형 -> 네트워크 NODE: 외부 RAW가 기존 NODE 속성으로 설명되는가

결과는 `walk_node_external_raw_overlap_summary.csv`에 저장합니다.

In [ ]:
NODE_REQUIRED_COLS = ["노드링크 유형", "노드 WKT", "노드 ID", "시군구명", "읍면동명", "횡단보도", "육교"]
missing_node_cols = [col for col in NODE_REQUIRED_COLS if col not in walk_raw.columns]
if missing_node_cols:
    print(f"walk_raw에 NODE 검증 컬럼이 없어 원본에서 다시 읽습니다: {missing_node_cols}")
    walk_nodes_raw = read_csv_auto(WALK_FILE, usecols=lambda col: col in NODE_REQUIRED_COLS)
    walk_nodes_raw = walk_nodes_raw[walk_nodes_raw["노드링크 유형"].eq("NODE")].copy()
else:
    walk_nodes_raw = walk_raw[walk_raw["노드링크 유형"].eq("NODE")].copy()
walk_nodes = gdf_from_wkt(walk_nodes_raw, "노드 WKT")
walk_nodes = walk_nodes[walk_nodes.geometry.geom_type.eq("Point")].copy()
walk_nodes = walk_nodes.rename(columns={"노드 ID": "walk_node_id"})
walk_nodes["walk_node_id"] = walk_nodes["walk_node_id"].astype(str)

NODE_VALIDATION_PAIRS = [
    {
        "attr_key": "crosswalk_node",
        "network_column": "횡단보도",
        "raw_keys": ["seoul_crosswalk"],
        "distance_m": 20,
    },
    {
        "attr_key": "overpass_node",
        "network_column": "육교",
        "raw_keys": ["seoul_overpass"],
        "distance_m": 20,
    },
]

for pair in NODE_VALIDATION_PAIRS:
    col = pair["network_column"]
    walk_nodes[f"is_{col}"] = to_flag(walk_nodes[col])


def validate_node_pair(pair: dict) -> tuple[dict, pd.DataFrame, pd.DataFrame]:
    attr_key = pair["attr_key"]
    network_col = pair["network_column"]
    flag_col = f"is_{network_col}"
    raw = combine_raw(pair["raw_keys"])
    distance_m = pair["distance_m"]

    flagged_nodes = walk_nodes[walk_nodes[flag_col]].copy()
    node_to_raw = sjoin_nearest_unique(
        flagged_nodes,
        raw,
        distance_m,
        "walk_node_id",
        ["raw_feature_id", "raw_dataset", "raw_dataset_label", "geometry_level"],
    )
    node_matched = node_to_raw["raw_feature_id"].notna()

    node_lookup = walk_nodes[["walk_node_id", flag_col, "시군구명", "읍면동명", "geometry"]].copy()
    raw_to_node = sjoin_nearest_unique(
        raw,
        node_lookup,
        distance_m,
        "raw_feature_id",
        ["walk_node_id", flag_col, "시군구명", "읍면동명"],
    )
    raw_near_any_node = raw_to_node["walk_node_id"].notna()
    raw_near_same_attr_node = raw_near_any_node & raw_to_node[flag_col].fillna(False)

    summary = {
        "attr_key": attr_key,
        "network_column": network_col,
        "raw_keys": ", ".join(pair["raw_keys"]),
        "distance_m": distance_m,
        "network_node_flag_count": len(flagged_nodes),
        "raw_feature_count": len(raw),
        "network_node_matched_count": int(node_matched.sum()),
        "network_node_match_rate": float(node_matched.mean()) if len(flagged_nodes) else np.nan,
        "raw_near_any_node_count": int(raw_near_any_node.sum()),
        "raw_to_network_node_attr_matched_count": int(raw_near_same_attr_node.sum()),
        "raw_to_network_node_attr_match_rate": float(raw_near_same_attr_node.mean()) if len(raw) else np.nan,
        "network_node_only_count": int((~node_matched).sum()),
        "raw_node_attr_missing_candidate_count": int((raw_near_any_node & ~raw_to_node[flag_col].fillna(False)).sum()),
        "raw_without_nearby_node_count": int((~raw_near_any_node).sum()),
    }

    unmatched_network_nodes = node_to_raw[node_to_raw["raw_feature_id"].isna()].copy()
    raw_node_attr_missing = raw_to_node[raw_near_any_node & ~raw_to_node[flag_col].fillna(False)].copy()
    unmatched_network_nodes["attr_key"] = attr_key
    raw_node_attr_missing["attr_key"] = attr_key
    return summary, unmatched_network_nodes, raw_node_attr_missing


node_summaries = []
node_only_samples = []
raw_node_attr_missing_samples = []

for pair in NODE_VALIDATION_PAIRS:
    summary, node_only, raw_missing = validate_node_pair(pair)
    node_summaries.append(summary)
    node_only_samples.append(node_only.head(200))
    raw_node_attr_missing_samples.append(raw_missing.head(200))

node_overlap_summary = pd.DataFrame(node_summaries)
node_only_sample = pd.concat(node_only_samples, ignore_index=True) if node_only_samples else pd.DataFrame()
raw_node_attr_missing_sample = pd.concat(raw_node_attr_missing_samples, ignore_index=True) if raw_node_attr_missing_samples else pd.DataFrame()

node_overlap_summary.to_csv(TABLE_DIR / "walk_node_external_raw_overlap_summary.csv", index=False, encoding="utf-8-sig")
node_only_sample.to_csv(TABLE_DIR / "walk_node_external_raw_node_only_sample.csv", index=False, encoding="utf-8-sig")
raw_node_attr_missing_sample.to_csv(TABLE_DIR / "walk_node_external_raw_attr_missing_candidate_sample.csv", index=False, encoding="utf-8-sig")

node_overlap_summary

## 7. 공원/녹지 point_proxy와 polygon 후보 점검

현재 로컬 RAW의 공원/녹지 후보는 대부분 대표점입니다. 대표점은 edge가 공원 내부인지 검증하기에는 약하므로, 먼저 거리별 민감도를 참고하고 polygon 파일이 로컬에 있는지 확인합니다.

- `walk_edge_green_point_proxy_sensitivity.csv`: 50m, 100m, 200m에서 공원/녹지 edge가 point_proxy와 얼마나 가까운지
- `park_polygon_candidate_inventory.csv`: 로컬에 공원/녹지 polygon으로 쓸 수 있는 파일이 있는지

In [ ]:
GREEN_POINT_RAW_KEYS = [
    "seoul_major_park",
    "national_city_park",
    "seoul_private_tree",
    "seoul_trail_line",
    "seoul_culture_trail_line",
]


def green_point_proxy_sensitivity(distances_m: list[int] = [50, 100, 200]) -> pd.DataFrame:
    green_edges = walk_edges[walk_edges["is_공원,녹지"]].copy()
    raw = combine_raw(GREEN_POINT_RAW_KEYS)
    rows = []
    for distance_m in distances_m:
        green_to_points = sjoin_nearest_unique(
            green_edges,
            raw,
            distance_m,
            "walk_edge_id",
            ["raw_feature_id", "raw_dataset", "raw_dataset_label", "geometry_level"],
        )
        matched = green_to_points["raw_feature_id"].notna()
        rows.append({
            "distance_m": distance_m,
            "network_green_edge_count": len(green_edges),
            "point_proxy_feature_count": len(raw),
            "green_edge_near_point_count": int(matched.sum()),
            "green_edge_near_point_rate": float(matched.mean()) if len(green_edges) else np.nan,
            "interpretation": "참고용: point_proxy 근접률이며 공원 내부 edge 검증은 아님",
        })
    return pd.DataFrame(rows)


green_point_sensitivity = green_point_proxy_sensitivity()
green_point_sensitivity.to_csv(TABLE_DIR / "walk_edge_green_point_proxy_sensitivity.csv", index=False, encoding="utf-8-sig")
green_point_sensitivity

In [ ]:
def inspect_polygon_candidate(path: Path) -> dict:
    sidecar_exts = [".shp", ".shx", ".dbf", ".prj", ".cpg"]
    sidecars = {ext: path.with_suffix(ext).exists() for ext in sidecar_exts} if path.suffix.lower() == ".shp" else {}
    row = {
        "file": path.name,
        "path": str(path),
        "sidecar_status": ", ".join([f"{ext}:{'Y' if ok else 'N'}" for ext, ok in sidecars.items()]),
        "readable_by_geopandas": False,
        "feature_count": np.nan,
        "geometry_types": "",
        "polygon_feature_count": np.nan,
        "note": "",
    }
    try:
        read_target = f"zip://{path.as_posix()}" if path.suffix.lower() == ".zip" else path
        gdf = gpd.read_file(read_target)
        row["readable_by_geopandas"] = True
        row["feature_count"] = len(gdf)
        row["geometry_types"] = ", ".join(sorted(gdf.geometry.geom_type.dropna().unique())) if len(gdf) else "empty"
        row["polygon_feature_count"] = int(gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"]).sum()) if len(gdf) else 0
    except Exception as exc:
        row["note"] = str(exc)[:200]
    return row


polygon_name_keywords = ["park", "green", "nature", "osm", "공원", "녹지", "upis", "zon216", "시설"]
polygon_extensions = {".geojson", ".json", ".gpkg", ".shp", ".zip"}
candidate_paths = [
    path
    for root in [RAW_DIR, PROJECT_ROOT / "analysis", PROJECT_ROOT / "cache"]
    if root.exists()
    for path in root.rglob("*")
    if path.is_file()
    and path.suffix.lower() in polygon_extensions
    and any(keyword.lower() in path.name.lower() for keyword in polygon_name_keywords)
]

if candidate_paths:
    park_polygon_candidate_inventory = pd.DataFrame([inspect_polygon_candidate(path) for path in candidate_paths])
else:
    park_polygon_candidate_inventory = pd.DataFrame([{
        "file": None,
        "path": None,
        "sidecar_status": None,
        "readable_by_geopandas": False,
        "feature_count": 0,
        "geometry_types": "none",
        "polygon_feature_count": 0,
        "note": "로컬 RAW/cache/analysis에서 공원·녹지 polygon 후보 파일을 찾지 못함. UPIS_SHP_ZON216.zip 같은 SHP zip을 src/data/raw에 넣거나 OSM polygon 수집 필요.",
    }])

park_polygon_candidate_inventory.to_csv(TABLE_DIR / "park_polygon_candidate_inventory.csv", index=False, encoding="utf-8-sig")
park_polygon_candidate_inventory

### 공원 polygon으로 실제 검증하기

`park_polygon_candidate_inventory`에서 `readable_by_geopandas=True`이고 `polygon_feature_count > 0`인 파일이 있으면, 도보 네트워크의 `공원,녹지` LINK가 polygon 안에 있거나 가까운지 검증합니다.

SHP는 `.shp`만 있으면 안 되고 보통 `.shx`, `.dbf`, `.prj`가 같은 폴더에 같이 있어야 합니다. `sidecar_status`에서 빠진 파일이 보이면 원본 zip을 풀 때 전체 파일을 같이 넣어야 합니다.

In [ ]:
def load_best_park_polygon() -> gpd.GeoDataFrame:
    if park_polygon_candidate_inventory.empty:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=WGS84)
    candidates = park_polygon_candidate_inventory[
        park_polygon_candidate_inventory["readable_by_geopandas"].eq(True)
        & park_polygon_candidate_inventory["polygon_feature_count"].fillna(0).gt(0)
    ].copy()
    if candidates.empty:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=WGS84)

    best_path = Path(candidates.sort_values("polygon_feature_count", ascending=False).iloc[0]["path"])
    read_target = f"zip://{best_path.as_posix()}" if best_path.suffix.lower() == ".zip" else best_path
    gdf = gpd.read_file(read_target)
    gdf = gdf[gdf.geometry.notna() & gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    if gdf.crs is None:
        print("주의: polygon CRS가 없습니다. 스크린샷 설명 기준 EPSG:5174로 가정합니다.")
        gdf = gdf.set_crs("EPSG:5174")
    gdf = gdf.to_crs(WGS84)
    gdf["park_polygon_source"] = best_path.name
    return gdf


park_polygons = load_best_park_polygon()
print(f"공원 polygon feature 수: {len(park_polygons):,}")
park_polygons.head()

### 공원 polygon 서울 범위/좌표계 확인

서울시 원천 데이터라도 실제 분석 전에 좌표계와 공간 범위를 확인합니다. 특히 EPSG가 잘못 잡히면 polygon이 서울 밖으로 튀거나 도보 네트워크와 전혀 겹치지 않습니다.

In [ ]:
def seoul_bbox_diagnostics(gdf: gpd.GeoDataFrame, label: str) -> pd.DataFrame:
    if len(gdf) == 0:
        return pd.DataFrame([{
            "label": label,
            "feature_count": 0,
            "crs": None,
            "min_lon": np.nan,
            "min_lat": np.nan,
            "max_lon": np.nan,
            "max_lat": np.nan,
            "centroid_in_seoul_bbox_count": 0,
            "centroid_in_seoul_bbox_rate": np.nan,
            "note": "empty geometry",
        }])
    wgs = gdf.to_crs(WGS84)
    bounds = wgs.total_bounds
    centroid = wgs.geometry.centroid
    in_bbox = centroid.x.between(126.7, 127.3) & centroid.y.between(37.35, 37.75)
    return pd.DataFrame([{
        "label": label,
        "feature_count": len(wgs),
        "crs": str(gdf.crs),
        "min_lon": bounds[0],
        "min_lat": bounds[1],
        "max_lon": bounds[2],
        "max_lat": bounds[3],
        "centroid_in_seoul_bbox_count": int(in_bbox.sum()),
        "centroid_in_seoul_bbox_rate": float(in_bbox.mean()),
        "note": "서울 bbox는 빠른 sanity check이며, 행정경계 정밀 판정은 별도 boundary polygon으로 확인",
    }])


park_polygon_seoul_bbox_check = seoul_bbox_diagnostics(park_polygons, "park_polygons")
park_polygon_seoul_bbox_check.to_csv(TABLE_DIR / "park_polygon_seoul_bbox_check.csv", index=False, encoding="utf-8-sig")
park_polygon_seoul_bbox_check

In [ ]:
def validate_green_edges_with_polygon(distances_m: list[int] = [0, 20, 50]) -> pd.DataFrame:
    green_edges = walk_edges[walk_edges["is_공원,녹지"]].copy()
    if len(park_polygons) == 0:
        return pd.DataFrame([{
            "distance_m": None,
            "network_green_edge_count": len(green_edges),
            "park_polygon_count": 0,
            "matched_green_edge_count": 0,
            "matched_green_edge_rate": np.nan,
            "note": "읽을 수 있는 공원 polygon 후보가 없음. SHP sidecar 파일(.shx/.dbf/.prj) 확인 필요.",
        }])

    green_m = green_edges[["walk_edge_id", "geometry"]].to_crs(METRIC_CRS)
    poly_m = park_polygons[["geometry"]].to_crs(METRIC_CRS)
    rows = []
    for distance_m in distances_m:
        target_poly = poly_m.copy()
        if distance_m > 0:
            target_poly["geometry"] = target_poly.geometry.buffer(distance_m)
        joined = gpd.sjoin(green_m, target_poly, how="left", predicate="intersects")
        matched_edge_count = joined.loc[joined["index_right"].notna(), "walk_edge_id"].nunique()
        rows.append({
            "distance_m": distance_m,
            "network_green_edge_count": len(green_edges),
            "park_polygon_count": len(park_polygons),
            "matched_green_edge_count": int(matched_edge_count),
            "matched_green_edge_rate": matched_edge_count / len(green_edges) if len(green_edges) else np.nan,
            "note": "distance_m=0은 polygon과 직접 교차/내부, 20/50m는 polygon 주변 buffer 포함",
        })
    return pd.DataFrame(rows)


green_polygon_validation = validate_green_edges_with_polygon()
green_polygon_validation.to_csv(TABLE_DIR / "walk_edge_green_polygon_validation.csv", index=False, encoding="utf-8-sig")
green_polygon_validation

## 8. 둘레길/문화길 보강 후보 검증

둘레길/문화길은 공원 polygon 검증 데이터가 아니라 별도 산책길 보강 후보입니다. 좌표가 서울 범위 안에 있는지, 도보 네트워크 edge와 가까운지, 기존 `공원,녹지` edge와 가까운지 따로 확인합니다.

In [ ]:
def load_trail_points(file_name: str, dataset_key: str, dataset_label: str, group_cols: list[str], order_col: str | None) -> gpd.GeoDataFrame:
    df = read_csv_auto(RAW_DIR / file_name)
    df["위도"] = pd.to_numeric(df["위도"], errors="coerce")
    df["경도"] = pd.to_numeric(df["경도"], errors="coerce")
    df = df.dropna(subset=["위도", "경도"]).copy()
    df["raw_dataset"] = dataset_key
    df["raw_dataset_label"] = dataset_label
    df["group_key"] = df[group_cols].fillna("").astype(str).agg(" | ".join, axis=1)
    df["order_value"] = pd.to_numeric(df[order_col], errors="coerce") if order_col and order_col in df.columns else np.arange(len(df))
    gdf = gdf_from_lon_lat(df, "경도", "위도")
    gdf["raw_feature_id"] = dataset_key + ":pt:" + gdf.index.astype(str)
    return gdf


trail_points = pd.concat([
    load_trail_points("서울시 둘레길 선형 위치정보 (좌표계_ WGS1984).csv", "seoul_trail_line_points", "서울시 둘레길 선형 좌표점", ["명칭"], "순번"),
    load_trail_points("서울시 문화길 선형 위치정보 (좌표계_ WGS1984).csv", "seoul_culture_trail_line_points", "서울시 문화길 선형 좌표점", ["파일경로", "명칭"], "고유번호"),
], ignore_index=True).pipe(gpd.GeoDataFrame, geometry="geometry", crs=WGS84)

trail_point_bbox_check = seoul_bbox_diagnostics(trail_points, "trail_culture_points")
trail_point_bbox_check.to_csv(TABLE_DIR / "trail_culture_point_seoul_bbox_check.csv", index=False, encoding="utf-8-sig")
trail_point_bbox_check

In [ ]:
def build_lines_from_points(points: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    records = []
    for (dataset, label, group_key), group in points.sort_values("order_value").groupby(["raw_dataset", "raw_dataset_label", "group_key"], dropna=False):
        coords = [(geom.x, geom.y) for geom in group.geometry if geom is not None]
        if len(coords) < 2:
            continue
        records.append({
            "raw_dataset": dataset.replace("_points", ""),
            "raw_dataset_label": label.replace(" 좌표점", " 선형후보"),
            "group_key": group_key,
            "point_count": len(coords),
            "geometry": LineString(coords),
        })
    if not records:
        return gpd.GeoDataFrame(columns=["raw_dataset", "raw_dataset_label", "group_key", "point_count", "geometry"], geometry="geometry", crs=WGS84)
    out = gpd.GeoDataFrame(records, geometry="geometry", crs=WGS84)
    out["raw_feature_id"] = out["raw_dataset"] + ":line:" + out.index.astype(str)
    return out


trail_lines = build_lines_from_points(trail_points)
trail_line_bbox_check = seoul_bbox_diagnostics(trail_lines, "trail_culture_lines")
trail_line_summary = trail_lines.groupby("raw_dataset", dropna=False).agg(
    line_count=("raw_feature_id", "count"),
    point_count_sum=("point_count", "sum"),
    point_count_median=("point_count", "median"),
).reset_index() if len(trail_lines) else pd.DataFrame(columns=["raw_dataset", "line_count", "point_count_sum", "point_count_median"])
trail_line_bbox_check.to_csv(TABLE_DIR / "trail_culture_line_seoul_bbox_check.csv", index=False, encoding="utf-8-sig")
trail_line_summary.to_csv(TABLE_DIR / "trail_culture_line_summary.csv", index=False, encoding="utf-8-sig")
display(trail_line_bbox_check)
trail_line_summary

In [ ]:
def nearest_match_summary(left: gpd.GeoDataFrame, right: gpd.GeoDataFrame, left_id: str, right_cols: list[str], distances_m: list[int], label: str) -> pd.DataFrame:
    rows = []
    for distance_m in distances_m:
        nearest = sjoin_nearest_unique(left, right, distance_m, left_id, right_cols)
        matched = nearest[right_cols[0]].notna()
        rows.append({
            "label": label,
            "distance_m": distance_m,
            "candidate_count": len(left),
            "matched_count": int(matched.sum()),
            "matched_rate": float(matched.mean()) if len(left) else np.nan,
        })
    return pd.DataFrame(rows)


walk_edge_lookup = walk_edges[["walk_edge_id", "시군구명", "읍면동명", "geometry"]].copy()
green_edge_lookup = walk_edges[walk_edges["is_공원,녹지"]][["walk_edge_id", "시군구명", "읍면동명", "geometry"]].copy()

trail_point_network_summary = pd.concat([
    nearest_match_summary(trail_points, walk_edge_lookup, "raw_feature_id", ["walk_edge_id", "시군구명", "읍면동명"], [20, 50, 100], "trail_points_to_any_walk_edge"),
    nearest_match_summary(trail_points, green_edge_lookup, "raw_feature_id", ["walk_edge_id", "시군구명", "읍면동명"], [20, 50, 100], "trail_points_to_green_edge"),
], ignore_index=True)

trail_point_network_summary.to_csv(TABLE_DIR / "trail_culture_point_network_overlap_summary.csv", index=False, encoding="utf-8-sig")
trail_point_network_summary

In [ ]:
if len(trail_lines):
    trail_line_network_summary = pd.concat([
        nearest_match_summary(trail_lines, walk_edge_lookup, "raw_feature_id", ["walk_edge_id", "시군구명", "읍면동명"], [20, 50, 100], "trail_lines_to_any_walk_edge"),
        nearest_match_summary(trail_lines, green_edge_lookup, "raw_feature_id", ["walk_edge_id", "시군구명", "읍면동명"], [20, 50, 100], "trail_lines_to_green_edge"),
    ], ignore_index=True)
else:
    trail_line_network_summary = pd.DataFrame(columns=["label", "distance_m", "candidate_count", "matched_count", "matched_rate"])

trail_line_network_summary.to_csv(TABLE_DIR / "trail_culture_line_network_overlap_summary.csv", index=False, encoding="utf-8-sig")
trail_line_network_summary

## 9. 자전거도로 active mobility 후보 검증

`전국자전거도로표준데이터`는 러닝길 확정 데이터가 아닙니다. 서울 필터 후 `자전거보행자겸용도로`, 하천/공원 주변, 도보 네트워크와 가까운 선형만 제한적으로 `active mobility` 또는 `riverside-linear-route` 후보로 봅니다.

확인할 것:

1. 서울 데이터만 남겼는가
2. 기점/종점 좌표가 있어 선형을 만들 수 있는가
3. `자전거도로종류`가 보행자 겸용인지
4. 도보 네트워크 edge와 가까운가
5. 공원/녹지 edge, 공원 polygon, 하천과 가까운가

In [59]:
def load_bike_road_candidates() -> gpd.GeoDataFrame:
    df = read_csv_auto(RAW_DIR / "전국자전거도로표준데이터.csv")
    df = df[df["시도명"].astype(str).str.contains("서울", na=False)].copy()
    coord_cols = ["기점위도", "기점경도", "종점위도", "종점경도"]
    for col in coord_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=coord_cols).copy()
    df["자전거도로종류"] = df["자전거도로종류"].fillna("미상").astype(str).str.strip()
    df["is_pedestrian_shared"] = df["자전거도로종류"].str.contains("보행자겸용|보행자", na=False)
    df["candidate_use"] = np.where(
        df["is_pedestrian_shared"],
        "active_mobility_shared_candidate",
        "bike_only_or_unclear_needs_review",
    )
    df["geometry"] = [
        LineString([(row["기점경도"], row["기점위도"]), (row["종점경도"], row["종점위도"])])
        for _, row in df.iterrows()
    ]
    gdf = gpd.GeoDataFrame(df, geometry="geometry", crs=WGS84)
    gdf["raw_dataset"] = "national_bike_road_seoul"
    gdf["raw_feature_id"] = "national_bike_road_seoul:" + gdf.index.astype(str)
    return gdf


bike_roads = load_bike_road_candidates()
bike_road_bbox_check = seoul_bbox_diagnostics(bike_roads, "national_bike_road_seoul")
bike_road_type_summary = bike_roads.groupby(["자전거도로종류", "candidate_use"], dropna=False).agg(
    feature_count=("raw_feature_id", "count"),
    total_length_km=("총길이(km)", lambda s: pd.to_numeric(s, errors="coerce").sum()),
).reset_index().sort_values("feature_count", ascending=False)
bike_road_bbox_check.to_csv(TABLE_DIR / "bike_road_seoul_bbox_check.csv", index=False, encoding="utf-8-sig")
bike_road_type_summary.to_csv(TABLE_DIR / "bike_road_type_summary.csv", index=False, encoding="utf-8-sig")
display(bike_road_bbox_check)
bike_road_type_summary

,label,feature_count,crs,min_lon,min_lat,max_lon,max_lat,centroid_in_seoul_bbox_count,centroid_in_seoul_bbox_rate,note
0,national_bike_road_seoul,188,EPSG:4326,126.822655,36.330655,127.401309,37.621909,187,0.994681,"서울 bbox는 빠른 sanity check이며, 행정경계 정밀 판정은 별도 bou..."


,자전거도로종류,candidate_use,feature_count,total_length_km
1,자전거보행자겸용도로,active_mobility_shared_candidate,109,95.180
3,자전거전용도로,bike_only_or_unclear_needs_review,42,36.880
0,미상,bike_only_or_unclear_needs_review,25,18.100
2,자전거우선도로,bike_only_or_unclear_needs_review,6,6.800
4,자전거전용차로,bike_only_or_unclear_needs_review,6,4.962


In [60]:
def bike_nearest_summary(distances_m: list[int] = [20, 50, 100]) -> pd.DataFrame:
    frames = []
    groups = {
        "bike_all": bike_roads,
        "bike_pedestrian_shared_only": bike_roads[bike_roads["is_pedestrian_shared"]].copy(),
    }
    targets = {
        "to_any_walk_edge": walk_edge_lookup,
        "to_green_edge": green_edge_lookup,
    }
    for group_label, group_gdf in groups.items():
        for target_label, target_gdf in targets.items():
            frames.append(nearest_match_summary(
                group_gdf,
                target_gdf,
                "raw_feature_id",
                ["walk_edge_id", "시군구명", "읍면동명"],
                distances_m,
                f"{group_label}_{target_label}",
            ))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


bike_road_network_overlap_summary = bike_nearest_summary()
bike_road_network_overlap_summary.to_csv(TABLE_DIR / "bike_road_network_overlap_summary.csv", index=False, encoding="utf-8-sig")
bike_road_network_overlap_summary

,label,distance_m,candidate_count,matched_count,matched_rate
0,bike_all_to_any_walk_edge,20,188,184,0.978723
1,bike_all_to_any_walk_edge,50,188,187,0.994681
2,bike_all_to_any_walk_edge,100,188,188,1.000000
3,bike_all_to_green_edge,20,188,17,0.090426
4,bike_all_to_green_edge,50,188,30,0.159574
5,bike_all_to_green_edge,100,188,47,0.250000
6,bike_pedestrian_shared_only_to_any_walk_edge,20,109,106,0.972477
7,bike_pedestrian_shared_only_to_any_walk_edge,50,109,108,0.990826
8,bike_pedestrian_shared_only_to_any_walk_edge,100,109,109,1.000000
9,bike_pedestrian_shared_only_to_green_edge,20,109,15,0.137615


In [61]:
def bike_context_summary(distance_m: int = 50) -> pd.DataFrame:
    rows = []
    groups = {
        "bike_all": bike_roads,
        "bike_pedestrian_shared_only": bike_roads[bike_roads["is_pedestrian_shared"]].copy(),
    }
    for group_label, group_gdf in groups.items():
        if len(group_gdf) == 0:
            rows.append({"label": group_label, "distance_m": distance_m, "context": "empty", "candidate_count": 0, "matched_count": 0, "matched_rate": np.nan})
            continue
        group_m = group_gdf[["raw_feature_id", "geometry"]].to_crs(METRIC_CRS)

        if len(park_polygons):
            poly_m = park_polygons[["geometry"]].to_crs(METRIC_CRS).copy()
            poly_m["geometry"] = poly_m.geometry.buffer(distance_m)
            joined = gpd.sjoin(group_m, poly_m, how="left", predicate="intersects")
            matched = joined.loc[joined["index_right"].notna(), "raw_feature_id"].nunique()
            rows.append({"label": group_label, "distance_m": distance_m, "context": "near_park_polygon", "candidate_count": len(group_gdf), "matched_count": int(matched), "matched_rate": matched / len(group_gdf)})

        river_path = RAW_DIR / "서울시 하천.geojson"
        if river_path.exists():
            rivers = gpd.read_file(river_path)
            if rivers.crs is None:
                rivers = rivers.set_crs(WGS84)
            river_m = rivers.to_crs(METRIC_CRS)[["geometry"]].copy()
            river_m["geometry"] = river_m.geometry.buffer(distance_m)
            joined = gpd.sjoin(group_m, river_m, how="left", predicate="intersects")
            matched = joined.loc[joined["index_right"].notna(), "raw_feature_id"].nunique()
            rows.append({"label": group_label, "distance_m": distance_m, "context": "near_river", "candidate_count": len(group_gdf), "matched_count": int(matched), "matched_rate": matched / len(group_gdf)})
    return pd.DataFrame(rows)


bike_road_context_summary = bike_context_summary(distance_m=50)
bike_road_context_summary.to_csv(TABLE_DIR / "bike_road_context_summary.csv", index=False, encoding="utf-8-sig")
bike_road_context_summary

,label,distance_m,context,candidate_count,matched_count,matched_rate
0,bike_all,50,near_park_polygon,188,84,0.446809
1,bike_all,50,near_river,188,12,0.063830
2,bike_pedestrian_shared_only,50,near_park_polygon,109,42,0.385321
3,bike_pedestrian_shared_only,50,near_river,109,9,0.082569


## 10. 나머지 RAW 검증/보강 후보 일괄 점검

아래 셀은 남은 RAW를 같은 점수로 바로 섞지 않고, 역할별로 분리해서 확인합니다.

- 기존 도보 네트워크 속성 검증용: 지하철역 연계 지하도, 자동차 전용도로, 공원 polygon, 횡단보도/육교/터널 RAW
- 새 보강 layer 후보: 보행자우선도로, 가로수길, 둘레길, 문화길, 자전거도로, 버스정류소, 엘리베이터/리프트, 화장실

여기서 나오는 값은 "바로 score 반영"이 아니라 `검증 가능`, `보강 후보`, `좌표/geometry 보류`를 가르는 근거입니다.


In [79]:
F_BUS = "서울시 버스정류소 위치정보.csv"
F_LIFT = "서울시 지하철 출입구 리프트 위치정보.csv"
F_ELEVATOR = "서울시 지하철역 엘리베이터 위치정보.csv"
F_UNDERGROUND = "서울시 지하철역 연계 지하도 공간정보.csv"
F_TREE = "전국가로수길정보표준데이터.csv"
F_CITY_PARK = "전국도시공원정보표준데이터.csv"
F_PED_PRIORITY = "전국보행자우선도로표준데이터.csv"
F_TOILET = "서울시 공중화장실 위치정보.csv"
F_CAR_ONLY = "서울시 자동차 전용도로 위치정보 (좌표계_ GRS80).csv"

C_NODE_WKT = "노드 WKT"
C_LINK_WKT = "링크 WKT"
C_SGG = "시군구명"
C_EMD = "읍면동명"
C_SIDO = "시도명"
C_LON = "경도"
C_LAT = "위도"
C_X = "X좌표"
C_Y = "Y좌표"

RAW_CANDIDATE_PLAN = [
    {"dataset": "bus_stop", "file": F_BUS, "role": "transit_access_candidate", "use": "bus stop POI; use carefully, not direct walk quality"},
    {"dataset": "subway_lift", "file": F_LIFT, "role": "accessibility_candidate", "use": "strong mobility-accessibility POI candidate"},
    {"dataset": "subway_elevator", "file": F_ELEVATOR, "role": "accessibility_candidate", "use": "strong mobility-accessibility POI candidate"},
    {"dataset": "subway_underground", "file": F_UNDERGROUND, "role": "network_attr_validation", "use": "validate subway/indoor/tunnel walk-network attrs"},
    {"dataset": "tree_road", "file": F_TREE, "role": "nature_scenic_candidate", "use": "shade/green/scenic line candidate after Seoul bbox and network overlap"},
    {"dataset": "national_city_park_point", "file": F_CITY_PARK, "role": "park_point_reference", "use": "secondary point reference because park polygon exists"},
    {"dataset": "pedestrian_priority_road", "file": F_PED_PRIORITY, "role": "walkability_safety_candidate", "use": "pedestrian-friendly road candidate"},
    {"dataset": "toilet", "file": F_TOILET, "role": "amenity_restroom_candidate", "use": "amenity POI; use coordinate file when available, otherwise geocoding"},
    {"dataset": "car_only_road", "file": F_CAR_ONLY, "role": "avoidance_or_exclusion_candidate", "use": "not positive; avoidance/barrier candidate, CRS first"},
]


def raw_file_exists(file_name: str) -> bool:
    return (RAW_DIR / file_name).exists()


def empty_candidate_gdf(dataset: str, label: str, geometry_level: str) -> gpd.GeoDataFrame:
    return gpd.GeoDataFrame(
        columns=["raw_dataset", "raw_dataset_label", "geometry_level", "raw_feature_id", "geometry"],
        geometry="geometry",
        crs=WGS84,
    )


def attach_candidate_metadata(gdf: gpd.GeoDataFrame, dataset: str, label: str, geometry_level: str) -> gpd.GeoDataFrame:
    if len(gdf) == 0:
        return empty_candidate_gdf(dataset, label, geometry_level)
    out = gdf.copy()
    out["raw_dataset"] = dataset
    out["raw_dataset_label"] = label
    out["geometry_level"] = geometry_level
    out["raw_feature_id"] = dataset + ":" + out.index.astype(str)
    return out


def filter_seoul_by_centroid_bbox(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if len(gdf) == 0:
        return gdf.copy()
    wgs = gdf.to_crs(WGS84)
    centroid = wgs.geometry.centroid
    mask = centroid.x.between(126.7, 127.3) & centroid.y.between(37.35, 37.75)
    return gdf.loc[mask.values].copy()


def point_gdf_from_lon_lat_cols(df: pd.DataFrame, lon_col: str, lat_col: str, crs=WGS84) -> gpd.GeoDataFrame:
    if lon_col not in df.columns or lat_col not in df.columns:
        return gpd.GeoDataFrame(columns=list(df.columns) + ["geometry"], geometry="geometry", crs=crs)
    return gdf_from_lon_lat(df, lon_col, lat_col, crs=crs)


def line_gdf_from_endpoint_cols(df: pd.DataFrame, start_lon: str, start_lat: str, end_lon: str, end_lat: str, source_crs=WGS84) -> gpd.GeoDataFrame:
    out = df.copy()
    coord_cols = [start_lon, start_lat, end_lon, end_lat]
    missing = [col for col in coord_cols if col not in out.columns]
    if missing:
        return gpd.GeoDataFrame(columns=list(df.columns) + ["geometry"], geometry="geometry", crs=source_crs)
    for col in coord_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out = out.dropna(subset=coord_cols).copy()
    if len(out) == 0:
        return gpd.GeoDataFrame(columns=list(df.columns) + ["geometry"], geometry="geometry", crs=source_crs)
    out["geometry"] = [
        LineString([(row[start_lon], row[start_lat]), (row[end_lon], row[end_lat])])
        for _, row in out.iterrows()
    ]
    return gpd.GeoDataFrame(out, geometry="geometry", crs=source_crs)


def nearest_candidate_summary(candidate: gpd.GeoDataFrame, target: gpd.GeoDataFrame, label: str, distances_m: list[int] = [20, 50, 100]) -> pd.DataFrame:
    if len(candidate) == 0 or len(target) == 0:
        return pd.DataFrame([{
            "label": label,
            "distance_m": d,
            "candidate_count": len(candidate),
            "matched_count": 0,
            "matched_rate": np.nan,
        } for d in distances_m])
    return nearest_match_summary(
        candidate,
        target,
        "raw_feature_id",
        ["walk_edge_id", C_SGG, C_EMD],
        distances_m,
        label,
    )


def summarize_candidate_inventory(candidates: dict[str, gpd.GeoDataFrame], notes: dict[str, str]) -> pd.DataFrame:
    rows = []
    plan_by_key = {row["dataset"]: row for row in RAW_CANDIDATE_PLAN}
    for key, plan in plan_by_key.items():
        gdf = candidates.get(key)
        if gdf is None:
            gdf = empty_candidate_gdf(key, plan["file"], "missing")
        bbox = seoul_bbox_diagnostics(gdf, key).iloc[0]
        rows.append({
            "dataset": key,
            "file": plan["file"],
            "role": plan["role"],
            "intended_use": plan["use"],
            "file_exists": raw_file_exists(plan["file"]),
            "geometry_level": ", ".join(sorted(gdf["geometry_level"].dropna().unique())) if "geometry_level" in gdf.columns and len(gdf) else "none",
            "feature_count": int(len(gdf)),
            "centroid_in_seoul_bbox_rate": bbox["centroid_in_seoul_bbox_rate"],
            "note": notes.get(key, ""),
        })
    return pd.DataFrame(rows)


In [80]:
# Transit/accessibility POI candidates: bus stops, subway lifts, subway elevators.
candidate_layers = {}
candidate_notes = {}

if raw_file_exists(F_BUS):
    bus_raw = read_csv_auto(RAW_DIR / F_BUS)
    bus_stop = point_gdf_from_lon_lat_cols(bus_raw, C_X, C_Y)
    bus_stop = filter_seoul_by_centroid_bbox(bus_stop)
    candidate_layers["bus_stop"] = attach_candidate_metadata(bus_stop, "bus_stop", F_BUS, "point_lon_lat")
    candidate_notes["bus_stop"] = "Transit access POI; not direct walk-quality score."
else:
    candidate_layers["bus_stop"] = empty_candidate_gdf("bus_stop", F_BUS, "missing")
    candidate_notes["bus_stop"] = "file missing"

if raw_file_exists(F_LIFT):
    lift_raw = read_csv_auto(RAW_DIR / F_LIFT)
    subway_lift = gdf_from_wkt(lift_raw, C_NODE_WKT)
    subway_lift = subway_lift[subway_lift.geometry.geom_type.eq("Point")].copy()
    subway_lift = filter_seoul_by_centroid_bbox(subway_lift)
    candidate_layers["subway_lift"] = attach_candidate_metadata(subway_lift, "subway_lift", F_LIFT, "point_wkt")
    candidate_notes["subway_lift"] = "Strong accessibility candidate; check proximity to walk edges."
else:
    candidate_layers["subway_lift"] = empty_candidate_gdf("subway_lift", F_LIFT, "missing")
    candidate_notes["subway_lift"] = "file missing"

if raw_file_exists(F_ELEVATOR):
    elevator_raw = read_csv_auto(RAW_DIR / F_ELEVATOR)
    subway_elevator = gdf_from_wkt(elevator_raw, C_NODE_WKT)
    subway_elevator = subway_elevator[subway_elevator.geometry.geom_type.eq("Point")].copy()
    subway_elevator = filter_seoul_by_centroid_bbox(subway_elevator)
    candidate_layers["subway_elevator"] = attach_candidate_metadata(subway_elevator, "subway_elevator", F_ELEVATOR, "point_wkt")
    candidate_notes["subway_elevator"] = "Strong accessibility candidate; check proximity to walk edges."
else:
    candidate_layers["subway_elevator"] = empty_candidate_gdf("subway_elevator", F_ELEVATOR, "missing")
    candidate_notes["subway_elevator"] = "file missing"

poi_accessibility_summary = pd.concat([
    nearest_candidate_summary(candidate_layers["bus_stop"], walk_edge_lookup, "bus_stop_to_any_walk_edge", [20, 50, 100]),
    nearest_candidate_summary(candidate_layers["subway_lift"], walk_edge_lookup, "subway_lift_to_any_walk_edge", [20, 50, 100]),
    nearest_candidate_summary(candidate_layers["subway_elevator"], walk_edge_lookup, "subway_elevator_to_any_walk_edge", [20, 50, 100]),
], ignore_index=True)
poi_accessibility_summary.to_csv(TABLE_DIR / "poi_accessibility_network_overlap_summary.csv", index=False, encoding="utf-8-sig")
poi_accessibility_summary


,label,distance_m,candidate_count,matched_count,matched_rate
0,bus_stop_to_any_walk_edge,20,11253,11195,0.994846
1,bus_stop_to_any_walk_edge,50,11253,11219,0.996979
2,bus_stop_to_any_walk_edge,100,11253,11237,0.998578
3,subway_lift_to_any_walk_edge,20,83,82,0.987952
4,subway_lift_to_any_walk_edge,50,83,83,1.000000
5,subway_lift_to_any_walk_edge,100,83,83,1.000000
6,subway_elevator_to_any_walk_edge,20,552,550,0.996377
7,subway_elevator_to_any_walk_edge,50,552,552,1.000000
8,subway_elevator_to_any_walk_edge,100,552,552,1.000000


In [72]:
# Subway underground links: validation candidate for subway-network, indoor, and tunnel attrs.
if raw_file_exists(F_UNDERGROUND):
    underground_raw = read_csv_auto(RAW_DIR / F_UNDERGROUND)
    underground_links = gdf_from_wkt(underground_raw, C_LINK_WKT)
    underground_links = underground_links[underground_links.geometry.geom_type.isin(["LineString", "MultiLineString"])].copy()
    underground_links = filter_seoul_by_centroid_bbox(underground_links)
    candidate_layers["subway_underground"] = attach_candidate_metadata(underground_links, "subway_underground", F_UNDERGROUND, "line_wkt")
    candidate_notes["subway_underground"] = "Compare underground lines with subway/indoor/tunnel network attrs."
else:
    candidate_layers["subway_underground"] = empty_candidate_gdf("subway_underground", F_UNDERGROUND, "missing")
    candidate_notes["subway_underground"] = "file missing"

underground_frames = [nearest_candidate_summary(candidate_layers["subway_underground"], walk_edge_lookup, "subway_underground_to_any_walk_edge", [20, 50, 100])]
for attr_key in ["subway", "indoor", "tunnel"]:
    col = SPECIAL_ATTRS[attr_key]["column"]
    target = walk_edges[walk_edges[f"is_{col}"]][["walk_edge_id", C_SGG, C_EMD, "geometry"]].copy()
    underground_frames.append(
        nearest_candidate_summary(
            candidate_layers["subway_underground"],
            target,
            f"subway_underground_to_network_{attr_key}_edge",
            [20, 50, 100],
        )
    )
underground_to_network_attr_summary = pd.concat(underground_frames, ignore_index=True)
underground_to_network_attr_summary.to_csv(TABLE_DIR / "subway_underground_network_attr_overlap_summary.csv", index=False, encoding="utf-8-sig")
underground_to_network_attr_summary


,label,distance_m,candidate_count,matched_count,matched_rate
0,subway_underground_to_any_walk_edge,20,3072,3051,0.993164
1,subway_underground_to_any_walk_edge,50,3072,3070,0.999349
2,subway_underground_to_any_walk_edge,100,3072,3072,1.000000
3,subway_underground_to_network_subway_edge,20,3072,80,0.026042
4,subway_underground_to_network_subway_edge,50,3072,147,0.047852
5,subway_underground_to_network_subway_edge,100,3072,219,0.071289
6,subway_underground_to_network_indoor_edge,20,3072,1582,0.514974
7,subway_underground_to_network_indoor_edge,50,3072,2202,0.716797
8,subway_underground_to_network_indoor_edge,100,3072,2724,0.886719
9,subway_underground_to_network_tunnel_edge,20,3072,18,0.005859


In [73]:
# Linear enhancement candidates: tree-lined roads and pedestrian-priority roads.
if raw_file_exists(F_TREE):
    tree_raw = read_csv_auto(RAW_DIR / F_TREE)
    tree_road = line_gdf_from_endpoint_cols(
        tree_raw,
        "가로수길시작경도", "가로수길시작위도",
        "가로수길종료경도", "가로수길종료위도",
    )
    tree_road = filter_seoul_by_centroid_bbox(tree_road)
    candidate_layers["tree_road"] = attach_candidate_metadata(tree_road, "tree_road", F_TREE, "line_from_endpoints")
    candidate_notes["tree_road"] = "Nationwide file; kept Seoul candidates by bbox. Shade/green/scenic candidate."
else:
    candidate_layers["tree_road"] = empty_candidate_gdf("tree_road", F_TREE, "missing")
    candidate_notes["tree_road"] = "file missing"

if raw_file_exists(F_PED_PRIORITY):
    ped_raw = read_csv_auto(RAW_DIR / F_PED_PRIORITY)
    if C_SIDO in ped_raw.columns:
        ped_raw = ped_raw[ped_raw[C_SIDO].astype(str).str.contains("서울", na=False)].copy()
    pedestrian_priority = line_gdf_from_endpoint_cols(
        ped_raw,
        "보행자우선도로시작점경도", "보행자우선도로시작점위도",
        "보행자우선도로종료점경도", "보행자우선도로종료점위도",
    )
    pedestrian_priority = filter_seoul_by_centroid_bbox(pedestrian_priority)
    candidate_layers["pedestrian_priority_road"] = attach_candidate_metadata(pedestrian_priority, "pedestrian_priority_road", F_PED_PRIORITY, "line_from_endpoints")
    candidate_notes["pedestrian_priority_road"] = "Walkability/safety enhancement candidate; use only near walk edges."
else:
    candidate_layers["pedestrian_priority_road"] = empty_candidate_gdf("pedestrian_priority_road", F_PED_PRIORITY, "missing")
    candidate_notes["pedestrian_priority_road"] = "file missing"

linear_candidate_network_overlap_summary = pd.concat([
    nearest_candidate_summary(candidate_layers["tree_road"], walk_edge_lookup, "tree_road_to_any_walk_edge", [20, 50, 100]),
    nearest_candidate_summary(candidate_layers["tree_road"], green_edge_lookup, "tree_road_to_green_edge", [20, 50, 100]),
    nearest_candidate_summary(candidate_layers["pedestrian_priority_road"], walk_edge_lookup, "pedestrian_priority_road_to_any_walk_edge", [20, 50, 100]),
], ignore_index=True)
linear_candidate_network_overlap_summary.to_csv(TABLE_DIR / "linear_candidate_network_overlap_summary.csv", index=False, encoding="utf-8-sig")
linear_candidate_network_overlap_summary


,label,distance_m,candidate_count,matched_count,matched_rate
0,tree_road_to_any_walk_edge,20,1558,774,0.496791
1,tree_road_to_any_walk_edge,50,1558,782,0.501926
2,tree_road_to_any_walk_edge,100,1558,790,0.507060
3,tree_road_to_green_edge,20,1558,72,0.046213
4,tree_road_to_green_edge,50,1558,118,0.075738
5,tree_road_to_green_edge,100,1558,165,0.105905
6,pedestrian_priority_road_to_any_walk_edge,20,122,122,1.000000
7,pedestrian_priority_road_to_any_walk_edge,50,122,122,1.000000
8,pedestrian_priority_road_to_any_walk_edge,100,122,122,1.000000


In [82]:
# National city-park points as secondary reference + toilet POI overlap.
def find_lon_lat_columns(df: pd.DataFrame) -> tuple[str | None, str | None]:
    lon_tokens = {"\uacbd\ub3c4", "x\uc88c\ud45c", "x", "lon", "lng", "longitude"}
    lat_tokens = {"\uc704\ub3c4", "y\uc88c\ud45c", "y", "lat", "latitude"}
    normalized = {col: str(col).strip().lower().replace(" ", "") for col in df.columns}

    lon_col = None
    lat_col = None
    for col, norm in normalized.items():
        if norm in lon_tokens or ("x" in norm and "\uc88c\ud45c" in norm) or norm.endswith("\uacbd\ub3c4"):
            lon_col = col
            break
    for col, norm in normalized.items():
        if norm in lat_tokens or ("y" in norm and "\uc88c\ud45c" in norm) or norm.endswith("\uc704\ub3c4"):
            lat_col = col
            break
    return lon_col, lat_col


if raw_file_exists(F_CITY_PARK):
    city_park_raw = read_csv_auto(RAW_DIR / F_CITY_PARK)
    city_park_point = point_gdf_from_lon_lat_cols(city_park_raw, C_LON, C_LAT)
    city_park_point = filter_seoul_by_centroid_bbox(city_park_point)
    candidate_layers["national_city_park_point"] = attach_candidate_metadata(city_park_point, "national_city_park_point", F_CITY_PARK, "point_lon_lat")
    candidate_notes["national_city_park_point"] = "Secondary point reference only because park polygon is available."
else:
    candidate_layers["national_city_park_point"] = empty_candidate_gdf("national_city_park_point", F_CITY_PARK, "missing")
    candidate_notes["national_city_park_point"] = "file missing"

city_park_point_summary = pd.concat([
    nearest_candidate_summary(candidate_layers["national_city_park_point"], walk_edge_lookup, "city_park_point_to_any_walk_edge", [20, 50, 100]),
    nearest_candidate_summary(candidate_layers["national_city_park_point"], green_edge_lookup, "city_park_point_to_green_edge", [20, 50, 100]),
], ignore_index=True)
city_park_point_summary.to_csv(TABLE_DIR / "city_park_point_network_overlap_summary.csv", index=False, encoding="utf-8-sig")

if raw_file_exists(F_TOILET):
    toilet_raw = read_csv_auto(RAW_DIR / F_TOILET)
    lon_col, lat_col = find_lon_lat_columns(toilet_raw)
    coord_like_cols = [col for col in toilet_raw.columns if any(token in str(col) for token in [C_LAT, C_LON, C_X, C_Y, "WKT", "x", "y"])]
    if lon_col and lat_col:
        toilet_point = point_gdf_from_lon_lat_cols(toilet_raw, lon_col, lat_col)
        toilet_point = filter_seoul_by_centroid_bbox(toilet_point)
        candidate_layers["toilet"] = attach_candidate_metadata(toilet_point, "toilet", F_TOILET, "point_lon_lat")
        toilet_network_overlap_summary = nearest_candidate_summary(candidate_layers["toilet"], walk_edge_lookup, "toilet_to_any_walk_edge", [20, 50, 100])
        geometry_usable_now = True
        decision = f"Coordinate file is usable with lon={lon_col}, lat={lat_col}. Treat as amenity/restroom POI candidate."
        candidate_notes["toilet"] = "Coordinate file usable; amenity/restroom POI candidate."
    else:
        candidate_layers["toilet"] = empty_candidate_gdf("toilet", F_TOILET, "address_only")
        toilet_network_overlap_summary = pd.DataFrame(columns=["label", "distance_m", "candidate_count", "matched_count", "matched_rate"])
        geometry_usable_now = False
        decision = "No usable coordinate columns. Hold until geocoding."
        candidate_notes["toilet"] = "No coordinate columns; spatial overlap not possible."
    toilet_quality_summary = pd.DataFrame([{
        "dataset": "toilet",
        "file": F_TOILET,
        "file_exists": True,
        "raw_feature_count": len(toilet_raw),
        "usable_feature_count": len(candidate_layers["toilet"]),
        "lon_col": lon_col,
        "lat_col": lat_col,
        "coordinate_like_columns": ", ".join(coord_like_cols),
        "geometry_usable_now": geometry_usable_now,
        "decision": decision,
    }])
else:
    candidate_layers["toilet"] = empty_candidate_gdf("toilet", F_TOILET, "missing")
    toilet_network_overlap_summary = pd.DataFrame(columns=["label", "distance_m", "candidate_count", "matched_count", "matched_rate"])
    toilet_quality_summary = pd.DataFrame([{
        "dataset": "toilet",
        "file": F_TOILET,
        "file_exists": False,
        "raw_feature_count": 0,
        "usable_feature_count": 0,
        "lon_col": None,
        "lat_col": None,
        "coordinate_like_columns": "",
        "geometry_usable_now": False,
        "decision": "file missing",
    }])
    candidate_notes["toilet"] = "file missing"

toilet_quality_summary.to_csv(TABLE_DIR / "toilet_quality_summary.csv", index=False, encoding="utf-8-sig")
toilet_network_overlap_summary.to_csv(TABLE_DIR / "toilet_network_overlap_summary.csv", index=False, encoding="utf-8-sig")
display(city_park_point_summary)
display(toilet_quality_summary)
toilet_network_overlap_summary


,label,distance_m,candidate_count,matched_count,matched_rate
0,city_park_point_to_any_walk_edge,20,3871,1256,0.324464
1,city_park_point_to_any_walk_edge,50,3871,1712,0.442263
2,city_park_point_to_any_walk_edge,100,3871,1775,0.458538
3,city_park_point_to_green_edge,20,3871,7,0.001808
4,city_park_point_to_green_edge,50,3871,28,0.007233
5,city_park_point_to_green_edge,100,3871,70,0.018083


,dataset,file,file_exists,raw_feature_count,usable_feature_count,lon_col,lat_col,coordinate_like_columns,geometry_usable_now,decision
0,toilet,서울시 공중화장실 위치정보.csv,True,4415,4413,x 좌표,y 좌표,"x 좌표, y 좌표",True,"Coordinate file is usable with lon=x 좌표, lat=y..."


,label,distance_m,candidate_count,matched_count,matched_rate
0,toilet_to_any_walk_edge,20,4413,3130,0.709268
1,toilet_to_any_walk_edge,50,4413,4235,0.959665
2,toilet_to_any_walk_edge,100,4413,4369,0.990029


In [77]:
# Car-only roads are not positive features. Diagnose coordinate usability first.
def car_only_crs_candidate_diagnostic(df: pd.DataFrame) -> pd.DataFrame:
    x = pd.to_numeric(df.get(C_X), errors="coerce")
    y = pd.to_numeric(df.get(C_Y), errors="coerce")
    valid = df[x.notna() & y.notna()].copy()
    rows = [{
        "crs_candidate": "raw_range",
        "feature_count": len(df),
        "valid_xy_count": len(valid),
        "min_lon": np.nan,
        "min_lat": np.nan,
        "max_lon": np.nan,
        "max_lat": np.nan,
        "centroid_in_seoul_bbox_count": 0,
        "centroid_in_seoul_bbox_rate": np.nan,
        "note": "Raw X/Y are not WGS84 lon/lat if outside 126-128, 36.5-38.5.",
        "x_min": float(x.min()) if x.notna().any() else np.nan,
        "x_max": float(x.max()) if x.notna().any() else np.nan,
        "y_min": float(y.min()) if y.notna().any() else np.nan,
        "y_max": float(y.max()) if y.notna().any() else np.nan,
    }]
    if len(valid) == 0:
        return pd.DataFrame(rows)
    raw_point = gpd.GeoDataFrame(valid.copy(), geometry=[Point(xy) for xy in zip(valid[C_X], valid[C_Y])])
    for epsg in [4326, 5174, 5178, 5179, 5181, 5186, 5187, 5188]:
        try:
            candidate = raw_point.set_crs(f"EPSG:{epsg}", allow_override=True).to_crs(WGS84)
            bbox = seoul_bbox_diagnostics(candidate, f"car_only_epsg_{epsg}").iloc[0]
            rows.append({
                "crs_candidate": f"EPSG:{epsg}",
                "feature_count": len(df),
                "valid_xy_count": len(valid),
                "min_lon": bbox["min_lon"],
                "min_lat": bbox["min_lat"],
                "max_lon": bbox["max_lon"],
                "max_lat": bbox["max_lat"],
                "centroid_in_seoul_bbox_count": bbox["centroid_in_seoul_bbox_count"],
                "centroid_in_seoul_bbox_rate": bbox["centroid_in_seoul_bbox_rate"],
                "note": "candidate CRS test; only use if bbox rate is high and visual sample passes",
                "x_min": float(x.min()) if x.notna().any() else np.nan,
                "x_max": float(x.max()) if x.notna().any() else np.nan,
                "y_min": float(y.min()) if y.notna().any() else np.nan,
                "y_max": float(y.max()) if y.notna().any() else np.nan,
            })
        except Exception as exc:
            rows.append({
                "crs_candidate": f"EPSG:{epsg}",
                "feature_count": len(df),
                "valid_xy_count": len(valid),
                "min_lon": np.nan,
                "min_lat": np.nan,
                "max_lon": np.nan,
                "max_lat": np.nan,
                "centroid_in_seoul_bbox_count": 0,
                "centroid_in_seoul_bbox_rate": np.nan,
                "note": f"transform failed: {exc}",
                "x_min": float(x.min()) if x.notna().any() else np.nan,
                "x_max": float(x.max()) if x.notna().any() else np.nan,
                "y_min": float(y.min()) if y.notna().any() else np.nan,
                "y_max": float(y.max()) if y.notna().any() else np.nan,
            })
    return pd.DataFrame(rows)


if raw_file_exists(F_CAR_ONLY):
    car_only_raw = read_csv_auto(RAW_DIR / F_CAR_ONLY)
    car_only_coordinate_diagnostic = car_only_crs_candidate_diagnostic(car_only_raw)
    best = car_only_coordinate_diagnostic[car_only_coordinate_diagnostic["crs_candidate"].ne("raw_range")].sort_values("centroid_in_seoul_bbox_rate", ascending=False).head(1)
    if len(best) and best["centroid_in_seoul_bbox_rate"].iloc[0] >= 0.8:
        best_epsg = best["crs_candidate"].iloc[0]
        car_only_point = point_gdf_from_lon_lat_cols(car_only_raw, C_X, C_Y, crs=best_epsg)
        car_only_point = car_only_point.to_crs(WGS84)
        car_only_point = filter_seoul_by_centroid_bbox(car_only_point)
        candidate_layers["car_only_road"] = attach_candidate_metadata(car_only_point, "car_only_road", F_CAR_ONLY, f"point_candidate_{best_epsg}")
        car_only_network_overlap_summary = nearest_candidate_summary(candidate_layers["car_only_road"], walk_edge_lookup, "car_only_road_to_any_walk_edge", [20, 50, 100])
        candidate_notes["car_only_road"] = f"Best CRS candidate is {best_epsg}; still use only for avoidance/barrier review after map sample check."
    else:
        candidate_layers["car_only_road"] = empty_candidate_gdf("car_only_road", F_CAR_ONLY, "crs_unknown")
        car_only_network_overlap_summary = pd.DataFrame(columns=["label", "distance_m", "candidate_count", "matched_count", "matched_rate"])
        candidate_notes["car_only_road"] = "No CRS candidate passed Seoul bbox check; hold spatial overlap until CRS/source geometry check."
else:
    car_only_coordinate_diagnostic = pd.DataFrame([{
        "crs_candidate": "missing",
        "feature_count": 0,
        "valid_xy_count": 0,
        "min_lon": np.nan,
        "min_lat": np.nan,
        "max_lon": np.nan,
        "max_lat": np.nan,
        "centroid_in_seoul_bbox_count": 0,
        "centroid_in_seoul_bbox_rate": np.nan,
        "note": "file missing",
        "x_min": np.nan,
        "x_max": np.nan,
        "y_min": np.nan,
        "y_max": np.nan,
    }])
    candidate_layers["car_only_road"] = empty_candidate_gdf("car_only_road", F_CAR_ONLY, "missing")
    car_only_network_overlap_summary = pd.DataFrame(columns=["label", "distance_m", "candidate_count", "matched_count", "matched_rate"])
    candidate_notes["car_only_road"] = "file missing"

car_only_coordinate_diagnostic.to_csv(TABLE_DIR / "car_only_coordinate_diagnostic.csv", index=False, encoding="utf-8-sig")
car_only_network_overlap_summary.to_csv(TABLE_DIR / "car_only_network_overlap_summary.csv", index=False, encoding="utf-8-sig")
display(car_only_coordinate_diagnostic)
car_only_network_overlap_summary


,crs_candidate,feature_count,valid_xy_count,min_lon,min_lat,max_lon,max_lat,centroid_in_seoul_bbox_count,centroid_in_seoul_bbox_rate,note,x_min,x_max,y_min,y_max
0,raw_range,168,168,NaN,NaN,NaN,NaN,0,NaN,Raw X/Y are not WGS84 lon/lat if outside 126-1...,17.27322,462222.298984,19.547101,40331.108787
1,EPSG:4326,168,168,17.273220,19.547101,462222.298984,40331.108787,0,0.0,candidate CRS test; only use if bbox rate is h...,17.27322,462222.298984,19.547101,40331.108787
2,EPSG:5174,168,168,124.849432,33.477916,129.831065,33.852602,0,0.0,candidate CRS test; only use if bbox rate is h...,17.27322,462222.298984,19.547101,40331.108787
3,EPSG:5178,168,168,117.990412,19.697425,122.356231,20.197262,0,0.0,candidate CRS test; only use if bbox rate is h...,17.27322,462222.298984,19.547101,40331.108787
4,EPSG:5179,168,168,117.992755,19.694660,122.358460,20.194508,0,0.0,candidate CRS test; only use if bbox rate is h...,17.27322,462222.298984,19.547101,40331.108787
5,EPSG:5181,168,168,124.848670,33.475148,129.830166,33.849820,0,0.0,candidate CRS test; only use if bbox rate is h...,17.27322,462222.298984,19.547101,40331.108787
6,EPSG:5186,168,168,124.870457,32.574106,129.801192,32.948451,0,0.0,candidate CRS test; only use if bbox rate is h...,17.27322,462222.298984,19.547101,40331.108787
7,EPSG:5187,168,168,126.870457,32.574106,131.801192,32.948451,0,0.0,candidate CRS test; only use if bbox rate is h...,17.27322,462222.298984,19.547101,40331.108787
8,EPSG:5188,168,168,128.870457,32.574106,133.801192,32.948451,0,0.0,candidate CRS test; only use if bbox rate is h...,17.27322,462222.298984,19.547101,40331.108787


,label,distance_m,candidate_count,matched_count,matched_rate


In [78]:
raw_candidate_inventory = summarize_candidate_inventory(candidate_layers, candidate_notes)
raw_candidate_inventory.to_csv(TABLE_DIR / "raw_candidate_role_inventory.csv", index=False, encoding="utf-8-sig")
raw_candidate_inventory


,dataset,file,role,intended_use,file_exists,geometry_level,feature_count,centroid_in_seoul_bbox_rate,note
0,bus_stop,서울시 버스정류소 위치정보.csv,transit_access_candidate,"bus stop POI; use carefully, not direct walk q...",True,point_lon_lat,11253,1.0,Transit access POI; not direct walk-quality sc...
1,subway_lift,서울시 지하철 출입구 리프트 위치정보.csv,accessibility_candidate,strong mobility-accessibility POI candidate,True,point_wkt,83,1.0,Strong accessibility candidate; check proximit...
2,subway_elevator,서울시 지하철역 엘리베이터 위치정보.csv,accessibility_candidate,strong mobility-accessibility POI candidate,True,point_wkt,552,1.0,Strong accessibility candidate; check proximit...
3,subway_underground,서울시 지하철역 연계 지하도 공간정보.csv,network_attr_validation,validate subway/indoor/tunnel walk-network attrs,True,line_wkt,3072,1.0,Compare underground lines with subway/indoor/t...
4,tree_road,전국가로수길정보표준데이터.csv,nature_scenic_candidate,shade/green/scenic line candidate after Seoul ...,True,line_from_endpoints,1558,1.0,Nationwide file; kept Seoul candidates by bbox...
5,national_city_park_point,전국도시공원정보표준데이터.csv,park_point_reference,secondary point reference because park polygon...,True,point_lon_lat,3871,1.0,Secondary point reference only because park po...
6,pedestrian_priority_road,전국보행자우선도로표준데이터.csv,walkability_safety_candidate,pedestrian-friendly road candidate,True,line_from_endpoints,122,1.0,Walkability/safety enhancement candidate; use ...
7,toilet,공중화장실정보_서울특별시.csv,amenity_restroom_candidate,amenity POI; use coordinate file when availabl...,True,none,0,NaN,No coordinate columns; spatial overlap not pos...
8,car_only_road,서울시 자동차 전용도로 위치정보 (좌표계_ GRS80).csv,avoidance_or_exclusion_candidate,"not positive; avoidance/barrier candidate, CRS...",True,none,0,NaN,No CRS candidate passed Seoul bbox check; hold...


## 11. 해석 체크리스트

- `network_flag_match_rate`가 높고 `raw_to_network_attr_match_rate`도 높으면 기존 네트워크 속성은 신뢰 근거가 생깁니다.
- `network_flag_match_rate`는 높은데 `raw_only_candidate_count`가 크면 외부 RAW가 보강 layer 후보입니다.
- `node_overlap_summary`는 횡단보도/육교의 NODE 속성 검증 결과입니다. LINK 결과와 섞어서 해석하지 않습니다.
- `trail_culture_*` 결과는 둘레길/문화길이 서울 안에 있고 도보 네트워크와 가까운지 보는 보강 후보 검증입니다. 공원 polygon 검증과 섞어서 해석하지 않습니다.
- `bike_road_*` 결과는 자전거도로를 러닝길로 확정하기 위한 것이 아니라, 보행자 겸용/도보 네트워크/공원/하천 맥락을 확인해 제한적 active mobility 후보로 분류하기 위한 것입니다.
- 둘 다 낮으면 좌표계, geometry 종류, 수집 범위, 속성 의미가 다를 가능성이 큽니다.
- `point_proxy`만 있는 RAW는 선형 edge 검증력이 낮으므로 `green_point_sensitivity`는 참고값으로만 보고, 공원 polygon/면 데이터 확보 전까지 보류 판정을 우선합니다.
- `raw_candidate_inventory`, `poi_accessibility_summary`, `underground_to_network_attr_summary`, `linear_candidate_network_overlap_summary`, `city_park_point_summary`, `toilet_quality_summary`, `toilet_network_overlap_summary`, `car_only_coordinate_diagnostic`은 나머지 RAW를 검증용/보강 후보/보류 대상으로 나눠 보기 위한 결과입니다.
- 결과 CSV는 `analysis/tables/raw/`에 저장됩니다.